# Zero-Shot CLIP Classification for Vector Features

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/clip_classify_vector.ipynb)

This notebook demonstrates how to classify vector polygon features using zero-shot CLIP inference with the `geoai.clip_classify` module. Given a set of polygons and a raster image, CLIP extracts an image chip for each polygon and matches it against user-provided category labels — no training data required.

## Key Features

- **Zero-shot classification**: Just provide candidate labels, no training needed
- **Automatic CRS alignment**: Vector and raster CRS are aligned automatically
- **Batch GPU inference**: Efficient batch processing with GPU acceleration
- **Top-k predictions**: Get multiple ranked predictions per polygon
- **Multiple export formats**: Save results as GeoJSON, GeoParquet, or GeoPackage

**Tip**: For best results with aerial/satellite imagery, use a remote-sensing CLIP model such as [`flax-community/clip-rsicd-v2`](https://huggingface.co/flax-community/clip-rsicd-v2), which was fine-tuned on remote sensing image captions.

## Install packages

In [ ]:
# %pip install geoai-py

## Import libraries

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from shapely.geometry import box

from geoai import clip_classify_vector, CLIPVectorClassifier, download_file

## Download sample data

We use a NAIP aerial image and building footprint polygons hosted on Hugging Face. The raster is a high-resolution (0.6 m) RGBN image covering a mixed-use area with buildings, vegetation, roads, and parking lots.

In [ ]:
raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train.tif"
)
vector_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train_buildings.geojson"

raster_path = download_file(raster_url)
vector_path = download_file(vector_url)

## Preview the data

Load the vector polygons and display the raster with overlaid building footprints.

In [ ]:
gdf = gpd.read_file(vector_path)
print(f"Number of building polygons: {len(gdf)}")
print(f"CRS: {gdf.crs}")
gdf.head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
with rasterio.open(raster_path) as src:
    show(src, ax=ax)
    gdf_proj = gdf.to_crs(src.crs)
gdf_proj.boundary.plot(ax=ax, color="red", linewidth=0.5)
ax.set_title("NAIP Image with Building Footprints")
plt.tight_layout()
plt.show()

## Land-cover classification with a grid of sample patches

To demonstrate zero-shot classification across diverse land-cover types, we create a regular grid of 60 m × 60 m polygons spanning the raster extent. Each grid cell captures a mix of buildings, vegetation, roads, and open areas — giving CLIP visually distinct categories to differentiate.

We use [`flax-community/clip-rsicd-v2`](https://huggingface.co/flax-community/clip-rsicd-v2), a CLIP model fine-tuned on remote sensing image captions (RSICD, UCM, Sydney datasets), which significantly outperforms the generic `openai/clip-vit-base-patch32` on aerial imagery.

In [ ]:
# Create a regular grid of 60 m x 60 m patches (100 x 100 pixels at 0.6 m)
with rasterio.open(raster_path) as src:
    left, bottom, right, top = src.bounds
    raster_crs = src.crs

cell_size = 60  # metres
grid_polys = []
for x in np.arange(left, right - cell_size, cell_size):
    for y in np.arange(bottom, top - cell_size, cell_size):
        grid_polys.append(box(x, y, x + cell_size, y + cell_size))

grid_gdf = gpd.GeoDataFrame(geometry=grid_polys, crs=raster_crs)
print(f"Grid polygons: {len(grid_gdf)}")

In [ ]:
labels = [
    "residential area",
    "dense vegetation",
    "road",
    "parking lot",
    "open grass field",
]

result_grid = clip_classify_vector(
    vector_data=grid_gdf,
    raster_path=raster_path,
    labels=labels,
    model_name="flax-community/clip-rsicd-v2",
)

## Explore results

The returned GeoDataFrame has two new columns:
- `clip_label` — the top-1 predicted category
- `clip_confidence` — the softmax confidence score (0 to 1)

In [ ]:
result_grid[["geometry", "clip_label", "clip_confidence"]].head(10)

In [ ]:
print("Label distribution:")
print(result_grid["clip_label"].value_counts())
print(f"\nMean confidence: {result_grid['clip_confidence'].mean():.3f}")

## Visualize classified grid

Color-code each grid cell by its predicted label and overlay on the raster.

In [ ]:
classified_grid = result_grid.dropna(subset=["clip_label"])

fig, ax = plt.subplots(1, 1, figsize=(12, 12))
with rasterio.open(raster_path) as src:
    show(src, ax=ax)

classified_grid.plot(
    column="clip_label",
    ax=ax,
    legend=True,
    alpha=0.5,
    edgecolor="black",
    linewidth=0.3,
    legend_kwds={"loc": "upper left", "fontsize": 8},
)
ax.set_title("Land-Cover Classification (RS-CLIP, 60 m grid)")
plt.tight_layout()
plt.show()

## Confidence scores

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
with rasterio.open(raster_path) as src:
    show(src, ax=ax)

classified_grid.plot(
    column="clip_confidence",
    ax=ax,
    legend=True,
    alpha=0.6,
    edgecolor="black",
    linewidth=0.3,
    cmap="RdYlGn",
    legend_kwds={"shrink": 0.5, "label": "Confidence"},
)
ax.set_title("CLIP Confidence Scores")
plt.tight_layout()
plt.show()

## Classify building footprints

We can also classify the original building footprints. Since all polygons are buildings, we use broader labels that CLIP can distinguish from roof appearance in aerial imagery.

In [ ]:
building_labels = [
    "residential building",
    "commercial building",
    "vegetation",
    "parking lot",
]

result_buildings = clip_classify_vector(
    vector_data=vector_path,
    raster_path=raster_path,
    labels=building_labels,
    model_name="flax-community/clip-rsicd-v2",
)

In [ ]:
print("Building classification distribution:")
print(result_buildings["clip_label"].value_counts())
print(f"\nMean confidence: {result_buildings['clip_confidence'].mean():.3f}")

In [ ]:
classified_bldg = result_buildings.dropna(subset=["clip_label"])
# Reproject to raster CRS for overlay
with rasterio.open(raster_path) as src:
    classified_bldg = classified_bldg.to_crs(src.crs)

fig, ax = plt.subplots(1, 1, figsize=(12, 12))
with rasterio.open(raster_path) as src:
    show(src, ax=ax)

classified_bldg.plot(
    column="clip_label",
    ax=ax,
    legend=True,
    alpha=0.5,
    edgecolor="black",
    linewidth=0.3,
    legend_kwds={"loc": "upper left", "fontsize": 8},
)
ax.set_title("Building Footprint Classification (RS-CLIP)")
plt.tight_layout()
plt.show()

## Top-k predictions with the class API

For more control, use `CLIPVectorClassifier` directly. Setting `top_k=3` returns the top 3 predictions per polygon, useful for understanding label ambiguity.

In [ ]:
classifier = CLIPVectorClassifier(model_name="flax-community/clip-rsicd-v2")

result_topk = classifier.classify(
    vector_data=grid_gdf,
    raster_path=raster_path,
    labels=labels,
    top_k=3,
    batch_size=32,
)

In [ ]:
# Show top-3 predictions for the first 5 classified polygons
classified = result_topk.dropna(subset=["clip_label"])
for i in range(min(5, len(classified))):
    row = classified.iloc[i]
    print(f"Polygon {classified.index[i]}:")
    for label, score in zip(row["clip_top_k_labels"], row["clip_top_k_scores"]):
        print(f"  {label:30s} {score:.3f}")
    print()

## Export results

Save the classified GeoDataFrame to GeoJSON, GeoParquet, or GeoPackage.

In [ ]:
result_grid.to_file("classified_landcover.geojson", driver="GeoJSON")
print("Results saved to classified_landcover.geojson")

## Summary

This notebook demonstrated:

1. **Zero-shot classification**: Classifying land-cover polygons with no training data — just candidate labels
2. **Remote-sensing CLIP model**: Using `flax-community/clip-rsicd-v2` for better aerial imagery understanding
3. **Grid-based land-cover mapping**: Creating sample patches to classify diverse land types
4. **Building footprint classification**: Applying CLIP to existing building polygons
5. **Top-k predictions**: Using `CLIPVectorClassifier` for ranked predictions
6. **Visualization**: Color-coded polygon maps and confidence score overlays
7. **Export**: Saving annotated results to standard geospatial formats

### Key Parameters

| Parameter | Description | Default |
|-----------|-------------|----------|
| `labels` | Candidate category names | (required) |
| `label_prefix` | Text prefix for CLIP encoding | `"a satellite image of "` |
| `model_name` | Hugging Face CLIP model ID | `"openai/clip-vit-base-patch32"` |
| `top_k` | Number of top predictions per polygon | `1` |
| `batch_size` | Images per inference batch | `16` |
| `min_chip_size` | Minimum chip dimension in pixels | `10` |
| `output_path` | Path to save results (`.geojson`, `.parquet`, `.gpkg`) | `None` |

### Model Recommendations

| Model | Best for | Notes |
|-------|----------|-------|
| `flax-community/clip-rsicd-v2` | Aerial / satellite imagery | Fine-tuned on RS image captions; ~88% top-1 accuracy on RSICD |
| `openai/clip-vit-base-patch32` | General-purpose images | Default CLIP; works best with ground-level photos |
| `openai/clip-vit-large-patch14` | Higher capacity | Larger model; slower but more expressive |